Gemma 4 E2B and E4B From Scratch (A Standalone Notebook with KV Cache)

In [ ]:
# 检查/打印本 notebook 依赖的关键第三方库版本,便于环境复现或排查依赖问题
# (相比很多其它 standalone 笔记本,这里多了 safetensors,用来直接读取单文件权重 checkpoint)
from importlib.metadata import version

pkgs = [
    "huggingface_hub",  # to download pretrained weights
    "safetensors",      # to load the checkpoint tensors
    "tokenizers",       # to implement the tokenizer
    "torch",            # to implement the model
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

In [ ]:
# 两个全局开关:
# CHOOSE_MODEL 选择 Gemma4 的稠密(Dense)模型规模——"E2B"(约 2B 有效参数)或 "E4B"(约 4B 有效参数),
# 对应下面 get_gemma4_dense_config() 里两套不同的层数/宽度/KV 共享比例配置;
# USE_INSTRUCT_MODEL 决定加载官方的指令微调(-it)版仓库,还是未微调的基础版仓库
CHOOSE_MODEL = "E2B"  # Options: "E2B", "E4B"
USE_INSTRUCT_MODEL = True

1. Architecture code

In [ ]:
# ===================== Gemma4 模型定义(含 KV Cache 支持) =====================
# 本单元格实现 Gemma4(稠密 Dense 变体,E2B / E4B)的核心组件:
#   - RoPE 旋转位置编码:局部滑窗层与全局层用不同频率基数;全局层还支持“部分旋转”
#     (rope_type="proportional"),即只旋转 head_dim 的一部分维度,其余维度不加位置编码(NoPE)
#   - Gemma4RMSNorm:均方根归一化,统计量强制在 float32 下计算;q_norm/k_norm 带可学习缩放,
#     v_norm 则是「无缩放」版本(with_scale=False),只做归一化不做可学习的逐维缩放
#   - Gemma4FeedForward:门控 MLP(GELU-tanh 近似,结构上类似 SwiGLU);当某一层处于
#     “KV 共享层”区间且开启 use_double_wide_mlp 时,中间隐藏维度翻倍,用额外的 MLP 容量
#     补偿该层不再自己算 KV 所省下的算力
#   - Gemma4Attention:分组查询注意力(GQA)+ Q/K/V 三者都过 RMSNorm;局部与全局注意力
#     使用不同的 head_dim;模型末尾若干层会直接复用前面同类型层算好的 K/V(“跨层 KV 共享”),
#     不再重复计算,以降低长上下文下的显存与算力开销
#   - Gemma4DenseBlock:局部滑窗注意力与全局注意力按层交替 + 双重 Post-Norm 残差结构,
#     并额外混入“逐层输入”(per-layer input,来自专属的逐层 embedding,详见 DenseModel)
#   - Gemma4DenseModel:词嵌入按 sqrt(emb_dim) 缩放、构造局部/全局两套注意力掩码、
#     管理逐层的 per-layer input 与 KV Cache、最终 logits 做 tanh soft-capping(软上限截断)
import torch
import torch.nn as nn


# ---- RoPE(旋转位置编码)参数预计算 ----
# 当 rope_type == "proportional" 时(Gemma4 的全局注意力层用此模式,见下方 rope_global_type):
# 只对 head_dim 的前 partial_rotary_factor 比例的维度施加旋转频率,其余维度的频率填 0,
# 相当于对剩下的维度完全不做位置编码(NoPE)。这样全局层里一部分通道随位置旋转编码相对位置,
# 另一部分通道则不携带位置信息,用于兼顾长距离建模能力与超长上下文外推时的稳定性
def compute_rope_params(
    head_dim,
    theta_base=10_000.0,
    context_length=4096,
    rope_type="default",
    partial_rotary_factor=1.0,
    dtype=torch.float32,
):
    if rope_type == "proportional":
        rope_angles = int(partial_rotary_factor * head_dim // 2)
        inv_freq_rotated = 1.0 / (
            theta_base ** (torch.arange(0, 2 * rope_angles, 2, dtype=torch.float32) / head_dim)
        )
        # inv_freq_rotated 形状: (rope_angles,),只覆盖 head_dim 的前 2*rope_angles 个维度对应的频率
        nope_angles = head_dim // 2 - rope_angles
        if nope_angles > 0:
            inv_freq = torch.cat([inv_freq_rotated, torch.zeros(nope_angles, dtype=torch.float32)], dim=0)
            # 用 0 频率(即 cos=1, sin=0,旋转角恒为 0)补齐剩余的 nope_angles 个维度,
            # 拼接后 inv_freq 形状恢复为 (head_dim // 2,),但后半部分对应“不旋转”的 NoPE 维度
        else:
            inv_freq = inv_freq_rotated
    else:
        inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        # rope_type=="default"(局部滑窗层使用):对全部 head_dim 维度按标准 RoPE 公式计算频率,
        # inv_freq 形状为 (head_dim // 2,),频率随维度指数衰减

    positions = torch.arange(context_length, dtype=torch.float32)
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)
    # positions 形状: (context_length,);angles 形状: (context_length, head_dim // 2),
    # 每个位置与每个频率两两相乘,得到该位置在该维度上的旋转角度
    angles = torch.cat([angles, angles], dim=1)
    # 把 angles 在最后一维复制一份拼接,得到 (context_length, head_dim),
    # 前后两半角度相同,配合 apply_rope 里“旋转一半维度”的写法使用
    cos = torch.cos(angles).to(dtype)
    sin = torch.sin(angles).to(dtype)
    return cos, sin
    # cos、sin 形状均为 (context_length, head_dim),预先算好整段可能用到的最大长度,
    # 实际使用时在 apply_rope 中按 [offset:offset+seq_len] 切片


# apply_rope:把预计算好的 cos/sin 应用到 q/k 张量上,实现旋转位置编码;
# offset 用于 KV Cache 场景——新 token 的绝对位置从 offset(已缓存的历史长度)开始,而不是永远从 0 开始
def apply_rope(x, cos, sin, offset=0):
    batch_size, num_heads, seq_len, head_dim = x.shape
    # x 形状: (batch_size, num_heads, seq_len, head_dim)
    assert head_dim % 2 == 0, "Head dimension must be even"

    x1 = x[..., : head_dim // 2]
    x2 = x[..., head_dim // 2 :]
    # x1、x2 形状均为 (batch_size, num_heads, seq_len, head_dim // 2),即把最后一维切成前后两半

    cos = cos[offset:offset + seq_len, :].unsqueeze(0).unsqueeze(0)
    sin = sin[offset:offset + seq_len, :].unsqueeze(0).unsqueeze(0)
    # 切片后 cos/sin 形状为 (seq_len, head_dim),unsqueeze 两次后变为 (1, 1, seq_len, head_dim),
    # 以便与 x 的 (batch_size, num_heads, seq_len, head_dim) 广播相乘
    rotated = torch.cat((-x2, x1), dim=-1)
    # rotated = (-x2, x1) 拼接,形状与 x 相同;等价于把 x 看作复数的实部/虚部做旋转(标准 RoPE 写法)
    return ((x * cos) + (rotated * sin)).to(dtype=x.dtype)


# repeat_kv:分组查询注意力(GQA)里,把 K/V 的头数从 num_kv_heads 扩展到 num_heads,
# 用 repeat_interleave 沿头维度重复,使每组 K/V 被 repeats(=num_heads // num_kv_heads)个 Query 头共享
def repeat_kv(x, repeats):
    if repeats == 1:
        return x
    return x.repeat_interleave(repeats, dim=1)
# ---- Gemma4RMSNorm:均方根归一化(RMSNorm),不做均值中心化,只按均方根缩放 ----
# with_scale=True(默认,用于 q_norm/k_norm 及各层的 layernorm)时带一个可学习的逐维缩放权重;
# with_scale=False(仅用于 v_norm)时不引入任何权重,纯粹对 value 做归一化,不做可学习缩放
class Gemma4RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6, with_scale=True):
        super().__init__()
        self.eps = eps
        self.with_scale = with_scale
        if with_scale:
            self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        x_float = x.float()
        # x_float 形状与 x 相同,只是转成 float32 计算,避免 bfloat16 精度不足导致方差估计不准
        mean_squared = x_float.pow(2).mean(dim=-1, keepdim=True) + self.eps
        # mean_squared 形状: (..., 1),对最后一维(emb_dim 或注意力里的 head_dim)求平方均值
        x_norm = x_float * torch.pow(mean_squared, -0.5)
        # x_norm 与 x 形状相同,乘以 mean_squared 的 -0.5 次方,即除以均方根(RMS),
        # 与 LayerNorm 不同的是这里不减均值,只做尺度归一化
        if self.with_scale:
            x_norm = x_norm * self.weight.float()
        return x_norm.to(dtype=x.dtype)


# ---- Gemma4FeedForward:门控 MLP(结构上类似 SwiGLU,用 GELU-tanh 近似作为门控激活) ----
class Gemma4FeedForward(nn.Module):
    def __init__(self, cfg, layer_idx):
        super().__init__()
        # Gemma4 把模型最后 num_kv_shared_layers 层设计成“跨层共享 KV”(自己不再算 K/V,
        # 直接复用前面某一层的结果,详见 Gemma4Attention.__init__ 中 kv_shared_layer_index 的计算)
        first_kv_shared_layer_idx = cfg["n_layers"] - cfg["num_kv_shared_layers"]
        is_kv_shared_layer = layer_idx >= first_kv_shared_layer_idx > 0
        use_double_wide_mlp = cfg["use_double_wide_mlp"] and is_kv_shared_layer
        intermediate_size = cfg["hidden_dim"] * (2 if use_double_wide_mlp else 1)
        # 若该层处于 KV 共享区间且开启 use_double_wide_mlp,则中间隐藏维度翻倍(intermediate_size),
        # 用更宽的 MLP 补偿该层因不再自己计算 K/V 而“节省”下来的算力,尽量不损失该层的建模能力
        self.gate_proj = nn.Linear(cfg["emb_dim"], intermediate_size, bias=False, dtype=cfg["dtype"])
        self.up_proj = nn.Linear(cfg["emb_dim"], intermediate_size, bias=False, dtype=cfg["dtype"])
        self.down_proj = nn.Linear(intermediate_size, cfg["emb_dim"], bias=False, dtype=cfg["dtype"])

    def forward(self, x):
        x_gate = self.gate_proj(x)
        x_up = self.up_proj(x)
        x = nn.functional.gelu(x_gate, approximate="tanh") * x_up
        # 门控机制: gelu_tanh(gate_proj(x)) * up_proj(x),两路都从 emb_dim 投影到 intermediate_size
        return self.down_proj(x)
# ==================== Gemma4Attention:分组查询注意力 + QKV-Norm + 局部/全局 + 跨层 KV 共享 ====================
# 要点:
#   1) 每一层的注意力类型(layer_type)由外部配置 layer_types 指定:"sliding_attention"(局部滑窗)
#      或 "full_attention"(全局);两者使用不同的 head_dim(局部用 head_dim,全局用 global_head_dim)
#   2) GQA: Query 有 num_heads 个头,Key/Value 只有 num_kv_heads 组,
#      每组被 num_key_value_groups = num_heads // num_kv_heads 个 Query 头共享,降低 KV Cache 显存占用
#   3) QKV-Norm: Q、K、V 在各自的 head_dim 维度上都过一次 RMSNorm(V 用无缩放版本),
#      是 Gemma3 QK-Norm 设计的进一步扩展
#   4) 跨层 KV 共享: 当该层位于模型最后 num_kv_shared_layers 层范围内时,
#      它不再自己计算 K/V,而是直接复用前面「最近一个同类型(sliding/full)层」算好的 K/V
class Gemma4Attention(nn.Module):
    def __init__(self, cfg, layer_idx):
        super().__init__()
        self.layer_type = cfg["layer_types"][layer_idx]
        self.is_sliding = self.layer_type == "sliding_attention"
        self.head_dim = cfg["head_dim"] if self.is_sliding else cfg["global_head_dim"]
        # 局部滑窗层用较小的 head_dim(如 256),全局层用独立配置的 global_head_dim(如 512),
        # 二者不要求相等,是 Gemma4 相较 Gemma3(局部/全局共用同一 head_dim)的一个区别
        self.num_heads = cfg["n_heads"]
        self.num_kv_heads = cfg["n_kv_heads"]
        self.num_key_value_groups = self.num_heads // self.num_kv_heads
        self.q_proj = nn.Linear(cfg["emb_dim"], self.num_heads * self.head_dim, bias=False, dtype=cfg["dtype"])
        self.k_proj = nn.Linear(cfg["emb_dim"], self.num_kv_heads * self.head_dim, bias=False, dtype=cfg["dtype"])
        self.v_proj = nn.Linear(cfg["emb_dim"], self.num_kv_heads * self.head_dim, bias=False, dtype=cfg["dtype"])
        self.o_proj = nn.Linear(self.num_heads * self.head_dim, cfg["emb_dim"], bias=False, dtype=cfg["dtype"])
        self.q_norm = Gemma4RMSNorm(self.head_dim, eps=cfg["layer_norm_eps"])
        self.k_norm = Gemma4RMSNorm(self.head_dim, eps=cfg["layer_norm_eps"])
        self.v_norm = Gemma4RMSNorm(self.head_dim, eps=cfg["layer_norm_eps"], with_scale=False)
        # q_norm/k_norm 带可学习缩放权重;v_norm 用 with_scale=False,即只做归一化、不做逐维缩放

        first_kv_shared_layer_idx = cfg["n_layers"] - cfg["num_kv_shared_layers"]
        self.is_kv_shared_layer = layer_idx >= first_kv_shared_layer_idx > 0
        prev_layers = cfg["layer_types"][:first_kv_shared_layer_idx]
        if self.is_kv_shared_layer:
            self.kv_shared_layer_index = len(prev_layers) - 1 - prev_layers[::-1].index(self.layer_type)
        else:
            self.kv_shared_layer_index = None
        # kv_shared_layer_index: 在“共享 KV 起点之前”的层里,从后往前找第一个与自己 layer_type 相同的层,
        # 记录其层号;forward 时若命中共享分支,就直接向 DenseModel 要那一层缓存/算好的 (key, value)

    # forward 参数说明:
    #   mask: 本层实际使用的注意力掩码(局部滑窗或全局,由 DenseBlock 按 layer_type 决定并传入)
    #   cos, sin: 对应局部或全局的 RoPE 预计算表
    #   start_pos: 本次前向新增 token 的起始绝对位置,用于 RoPE 与（重新拼接后的）KV Cache 对齐
    #   cache: 该层自身的 (key, value) 历史缓存;shared_kv: 若非 None,直接复用该 (key, value),
    #          不再自己计算/更新缓存(对应上面 kv_shared_layer_index 指向的那一层)
    def forward(self, x, mask, cos, sin, start_pos=0, cache=None, shared_kv=None):
        batch_size, seq_len, _ = x.shape
        # x 形状: (batch_size, seq_len, emb_dim);seq_len 是本次前向新增的 token 数
        query = self.q_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        # 投影后 reshape 成多头再 transpose,得到 query 形状: (batch_size, num_heads, seq_len, head_dim)
        query = self.q_norm(query)
        query = apply_rope(query, cos, sin, offset=start_pos)

        if shared_kv is None:
            key = self.k_proj(x).view(batch_size, seq_len, self.num_kv_heads, self.head_dim).transpose(1, 2)
            value = self.v_proj(x).view(batch_size, seq_len, self.num_kv_heads, self.head_dim).transpose(1, 2)
            key = self.k_norm(key)
            value = self.v_norm(value)
            key = apply_rope(key, cos, sin, offset=start_pos)
            # 注意:key 在与历史 cache 拼接之前就先算好了自身的 RoPE(用 start_pos 作为该段的起始位置);
            # value 则不做任何位置编码(RoPE 只作用于 Q/K,不作用于 V,这是标准做法)

            if cache is not None and cache[0] is not None:
                key = torch.cat([cache[0], key], dim=2)
                value = torch.cat([cache[1], value], dim=2)
                # 沿 dim=2(seq/时间维)拼接历史与新算出的 key/value,
                # 拼接后形状为 (batch_size, num_kv_heads, cached_len + seq_len, head_dim)

            next_cache = (key, value)
            # next_cache 是本层更新后的完整 (key, value),交由 DenseModel 写回该层的 KV Cache
        else:
            key, value = shared_kv
            next_cache = None
            # 命中跨层共享:直接用调用方传入的 (key, value)(来自另一层),本层不产生新的 cache 需要写回

        key_for_attn = repeat_kv(key, self.num_key_value_groups)
        value_for_attn = repeat_kv(value, self.num_key_value_groups)
        # key_for_attn / value_for_attn 形状: (batch_size, num_heads, kv_len, head_dim),
        # 通过 repeat_kv 把 K/V 头数从 num_kv_heads 广播到 num_heads,与 query 的头数对齐

        attn_scores = query @ key_for_attn.transpose(-1, -2)
        # attn_scores 形状: (batch_size, num_heads, seq_len, kv_len)
        # 风险标注:此处未对 attn_scores 做 1/sqrt(head_dim) 等显式缩放(对比同目录 Gemma3 版本
        # 的 standalone 实现,那里用 query_pre_attn_scalar 显式缩放了 query)。这里保持原代码逻辑不变,
        # 是否等价于官方 Gemma4 实现(例如缩放被隐式折叠进了 q_norm/k_norm 的可学习权重里)未做验证,
        # 仅作风险提示,不在本次任务范围内修改
        attn_scores = attn_scores.masked_fill(mask.unsqueeze(0).unsqueeze(0), torch.finfo(attn_scores.dtype).min)
        # mask 为 True 的位置表示“不可见”(未来位置,或超出滑窗范围),用该 dtype 的最小值填充后再 softmax
        attn_weights = torch.softmax(attn_scores.float(), dim=-1).to(dtype=query.dtype)
        # softmax 在 float32 下计算以保证数值稳定,算完再转回 query 的原始 dtype(如 bfloat16)
        context = attn_weights @ value_for_attn
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.num_heads * self.head_dim)
        # context 从 (batch_size, num_heads, seq_len, head_dim) 变回 (batch_size, seq_len, num_heads*head_dim)
        output = self.o_proj(context)
        return output, next_cache
# ==================== Gemma4DenseBlock:局部/全局注意力交替 + 双重 Post-Norm + 逐层输入混合 ====================
# 归一化结构与 Gemma3 相同,是 Pre-Norm 与 Post-Norm 的结合(而非 GPT 系列的纯 Pre-Norm):
#   input_layernorm -> Attention -> post_attention_layernorm -> 残差相加
#   pre_feedforward_layernorm -> FeedForward -> post_feedforward_layernorm -> 残差相加
# 在此基础上,Gemma4 额外加入了“逐层输入”(per-layer input)混合步骤(hidden_size_per_layer_input>0 时启用),
# 这是 Gemma3n 系列引入的“逐层专属 embedding”设计的延续,让每一层都能直接看到一份
# 与词元 id 相关、但独立于主干隐藏状态演化的额外信息
class Gemma4DenseBlock(nn.Module):
    def __init__(self, cfg, layer_idx):
        super().__init__()
        self.layer_idx = layer_idx
        self.layer_type = cfg["layer_types"][layer_idx]
        self.sliding_window = cfg["sliding_window"]
        self.att = Gemma4Attention(cfg, layer_idx)
        self.mlp = Gemma4FeedForward(cfg, layer_idx)
        self.input_layernorm = Gemma4RMSNorm(cfg["emb_dim"], eps=cfg["layer_norm_eps"])
        self.post_attention_layernorm = Gemma4RMSNorm(cfg["emb_dim"], eps=cfg["layer_norm_eps"])
        self.pre_feedforward_layernorm = Gemma4RMSNorm(cfg["emb_dim"], eps=cfg["layer_norm_eps"])
        self.post_feedforward_layernorm = Gemma4RMSNorm(cfg["emb_dim"], eps=cfg["layer_norm_eps"])
        self.register_buffer("layer_scalar", torch.ones(1), persistent=True)
        # layer_scalar: 一个初始为全 1 的标量 buffer,forward 末尾会乘到该层输出上;
        # 从权重加载单元格可见它也会被预训练权重覆盖(见 load_weights_into_gemma4_dense 中的 layer_scalar 赋值)
        self.hidden_size_per_layer_input = cfg["hidden_size_per_layer_input"]
        if self.hidden_size_per_layer_input:
            self.per_layer_input_gate = nn.Linear(
                cfg["emb_dim"],
                self.hidden_size_per_layer_input,
                bias=False,
                dtype=cfg["dtype"],
            )
            self.per_layer_projection = nn.Linear(
                self.hidden_size_per_layer_input,
                cfg["emb_dim"],
                bias=False,
                dtype=cfg["dtype"],
            )
            self.post_per_layer_input_norm = Gemma4RMSNorm(cfg["emb_dim"], eps=cfg["layer_norm_eps"])
            # per_layer_input_gate: emb_dim -> hidden_size_per_layer_input
            # per_layer_projection:  hidden_size_per_layer_input -> emb_dim(先降维再升维回主干宽度)

    def forward(
        self,
        x,
        per_layer_input,
        mask_local,
        mask_global,
        cos_local,
        sin_local,
        cos_global,
        sin_global,
        start_pos=0,
        cache=None,
        shared_kv=None,
    ):
        if self.layer_type == "sliding_attention":
            mask = mask_local
            cos = cos_local
            sin = sin_local
        else:
            mask = mask_global
            cos = cos_global
            sin = sin_global
        # 根据本层类型选择对应的注意力掩码与 RoPE 频率表(局部滑窗层用 mask_local + cos_local/sin_local,
        # 全局层用 mask_global + cos_global/sin_global)

        if shared_kv is not None:
            eff_kv_len = shared_kv[0].size(2)
        elif cache is not None and cache[0] is not None:
            eff_kv_len = cache[0].size(2) + x.size(1)
        else:
            eff_kv_len = x.size(1)
        mask = mask[..., -eff_kv_len:]
        # mask 原本是针对“完整位置区间”预先构造好的,这里裁掉多余的历史列,
        # 使掩码的最后一维宽度与本层实际参与注意力计算的 K 长度(eff_kv_len)对齐

        residual = x
        x = self.input_layernorm(x)
        x_attn, next_cache = self.att(
            x,
            mask,
            cos,
            sin,
            start_pos=start_pos,
            cache=cache,
            shared_kv=shared_kv,
        )
        if next_cache is not None and self.layer_type == "sliding_attention":
            key, value = next_cache
            if key.size(2) > self.sliding_window:
                key = key[:, :, -self.sliding_window:, :]
                value = value[:, :, -self.sliding_window:, :]
            next_cache = (key, value)
            # 滑窗层专属:注意力算完后若该层 KV Cache 长度超过 sliding_window,
            # 只保留最近 sliding_window 个 token,让局部层显存占用恒定,不随生成长度无限增长

        x_attn = self.post_attention_layernorm(x_attn)
        x = residual + x_attn

        residual = x
        x = self.pre_feedforward_layernorm(x)
        x = self.mlp(x)
        x = self.post_feedforward_layernorm(x)
        x = residual + x

        if self.hidden_size_per_layer_input:
            residual = x
            x_per_layer = self.per_layer_input_gate(x)
            x_per_layer = nn.functional.gelu(x_per_layer, approximate="tanh")
            x_per_layer = x_per_layer * per_layer_input
            x_per_layer = self.per_layer_projection(x_per_layer)
            x_per_layer = self.post_per_layer_input_norm(x_per_layer)
            x = residual + x_per_layer
            # 把逐层输入(per_layer_input,来自 DenseModel 里预先算好的逐层 embedding)
            # 经过门控 + 逐维相乘 + 投影回 emb_dim + 归一化后,以残差形式加回主干隐藏状态

        return x * self.layer_scalar.to(dtype=x.dtype), next_cache
        # 最终输出再乘以 layer_scalar(初始为 1,可能被预训练权重覆盖为其它标量)
# ==================== Gemma4DenseModel:整体模型(词嵌入缩放 + 逐层前向 + KV Cache 管理) ====================
class Gemma4DenseModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg["layer_types"] is not None and len(cfg["layer_types"]) == cfg["n_layers"]
        self.cfg = cfg
        self.tok_emb = nn.Embedding(
            cfg["vocab_size"],
            cfg["emb_dim"],
            padding_idx=cfg.get("pad_token_id", 0),
            dtype=cfg["dtype"],
        )
        # tok_emb 权重形状: (vocab_size, emb_dim)
        self.blocks = nn.ModuleList([Gemma4DenseBlock(cfg, i) for i in range(cfg["n_layers"])])
        self.final_norm = Gemma4RMSNorm(cfg["emb_dim"], eps=cfg["layer_norm_eps"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False, dtype=cfg["dtype"])
        if cfg.get("tie_word_embeddings", False):
            self.out_head.weight = self.tok_emb.weight
            # 权重绑定(weight tying):输出层直接复用词嵌入矩阵,不单独存一份 lm_head 权重
        self.current_pos = 0
        # current_pos: 记录到目前为止已经喂给模型的 token 总数,供 KV Cache 增量推理时作为下一次的起始位置

        self.hidden_size_per_layer_input = cfg["hidden_size_per_layer_input"]
        if self.hidden_size_per_layer_input:
            self.embed_tokens_per_layer = nn.Embedding(
                cfg["vocab_size_per_layer_input"],
                cfg["n_layers"] * self.hidden_size_per_layer_input,
                padding_idx=cfg.get("pad_token_id", 0),
                dtype=cfg["dtype"],
            )
            self.per_layer_model_projection = nn.Linear(
                cfg["emb_dim"],
                cfg["n_layers"] * self.hidden_size_per_layer_input,
                bias=False,
                dtype=cfg["dtype"],
            )
            self.per_layer_projection_norm = Gemma4RMSNorm(
                self.hidden_size_per_layer_input,
                eps=cfg["layer_norm_eps"],
            )
            # embed_tokens_per_layer: 每个 token id 对应一份形状为 (n_layers * hidden_size_per_layer_input,)
            # 的“逐层专属”embedding,forward 时会 reshape 成 (..., n_layers, hidden_size_per_layer_input)
            # 并按层号切片喂给对应的 DenseBlock,即前面 Gemma4DenseBlock 中提到的 per_layer_input

        rope_local_type = cfg.get("rope_local_type", "default")
        cos_local, sin_local = compute_rope_params(
            head_dim=cfg["head_dim"],
            theta_base=cfg["rope_local_base"],
            context_length=cfg["context_length"],
            rope_type=rope_local_type,
            dtype=torch.float32,
        # 局部滑窗层 RoPE 表:使用较小的 rope_local_base(如 10_000),偏向短距离位置关系,
        # rope_type 默认为 "default",即对全部 head_dim 维度都施加旋转
        )
        cos_global, sin_global = compute_rope_params(
            head_dim=cfg["global_head_dim"],
            theta_base=cfg["rope_global_base"],
            context_length=cfg["context_length"],
            rope_type=cfg["rope_global_type"],
            partial_rotary_factor=cfg["rope_global_partial_rotary_factor"],
            dtype=torch.float32,
        )
        # 全局层 RoPE 表:使用较大的 rope_global_base(如 1_000_000),并以 "proportional" 模式
        # 只旋转 rope_global_partial_rotary_factor(如 0.25)比例的维度,其余维度不做位置编码(NoPE),
        # 是 Gemma4 长上下文设计中“旋转 + 非旋转”混合的具体体现
        self.register_buffer("cos_local", cos_local, persistent=False)
        self.register_buffer("sin_local", sin_local, persistent=False)
        self.register_buffer("cos_global", cos_global, persistent=False)
        self.register_buffer("sin_global", sin_global, persistent=False)

    # _create_masks: 一次性构造“全局因果掩码”和“局部滑窗掩码”,两者都基于 [0, pos_end) 的完整位置区间;
    # 使用 KV Cache 时,通过 pos_start/pos_end 只截取当前 step 需要的“query 行”,
    # 但“key 列”仍覆盖到 pos_end(即全部历史 key),再由 DenseBlock.forward 按 eff_kv_len 做进一步裁剪
    def _create_masks(self, cur_len, device, pos_start=0, pos_end=None):
        if pos_end is None:
            pos_end = cur_len

        ones = torch.ones((pos_end, pos_end), dtype=torch.bool, device=device)
        # ones / 掩码矩阵形状均为 (pos_end, pos_end)
        mask_global_full = torch.triu(ones, diagonal=1)
        # mask_global_full: 标准因果掩码,True 表示“未来位置,不可见”(上三角,不含对角线)
        far_past_full = torch.triu(ones, diagonal=self.cfg["sliding_window"]).T
        # far_past_full: True 表示“距离当前位置太远的历史(超出 sliding_window)”,
        # 转置后落在下三角区域,即滑窗层需要额外屏蔽掉的“太久远的过去”
        mask_local_full = mask_global_full | far_past_full
        # mask_local_full = 因果屏蔽 或 滑窗外屏蔽,两者取并集,即滑窗层能看到的只有
        # [当前位置 - sliding_window + 1, 当前位置] 这一小段窗口内的历史 token

        row_slice = slice(pos_start, pos_end)
        mask_global = mask_global_full[row_slice, :pos_end]
        mask_local = mask_local_full[row_slice, :pos_end]
        return mask_global, mask_local
        # 返回的 mask_global / mask_local 形状均为 (pos_end - pos_start, pos_end)

    # get_per_layer_inputs: 按 token id 查出每层专属的逐层 embedding,并按 sqrt(hidden_size_per_layer_input) 缩放
    def get_per_layer_inputs(self, input_ids):
        if not self.hidden_size_per_layer_input:
            return None
        return (self.embed_tokens_per_layer(input_ids) * (self.hidden_size_per_layer_input ** 0.5)).reshape(
            *input_ids.shape,
            self.cfg["n_layers"],
            self.hidden_size_per_layer_input,
        )
        # 返回形状: (*input_ids.shape, n_layers, hidden_size_per_layer_input)

    # project_per_layer_inputs: 把主干隐藏状态 inputs_embeds 投影成“每层一份”的向量,
    # 再与 get_per_layer_inputs 得到的逐层 embedding 相加融合(各自缩放后按 (a+b)/sqrt(2) 平均),
    # 使每一层同时拿到“来自主干的信息”与“词元专属的逐层信息”两部分输入
    def project_per_layer_inputs(self, inputs_embeds, per_layer_inputs=None):
        if not self.hidden_size_per_layer_input:
            return None
        projected = self.per_layer_model_projection(inputs_embeds) * (self.cfg["emb_dim"] ** -0.5)
        projected = projected.reshape(
            *inputs_embeds.shape[:-1],
            self.cfg["n_layers"],
            self.hidden_size_per_layer_input,
        )
        projected = self.per_layer_projection_norm(projected)
        if per_layer_inputs is None:
            return projected
        return (projected + per_layer_inputs) * (2.0 ** -0.5)
        # (2.0 ** -0.5) 相当于把两路信号按等权重平均后再乘 sqrt(2)/2,保持整体方差稳定

    # Gemma4DenseModel.forward: 单次前向,可选地读写 KV Cache
    #   - cache 为 None: 常规一次性前向(如一次性计算整段 prompt 的 logits)
    #   - cache 不为 None: 增量推理模式,每次只传入“新增”的 token(首次是整段 prompt,之后每次 1 个 token)
    def forward(self, input_ids, cache=None):
        x = self.tok_emb(input_ids) * (self.cfg["emb_dim"] ** 0.5)
        # 关键点:词嵌入之后要乘以 sqrt(emb_dim) 做缩放,是 Gemma 系列的标志性设计之一,
        # 让嵌入向量的方差与其它层激活值处于同一量级
        per_layer_inputs = self.get_per_layer_inputs(input_ids)
        per_layer_inputs = self.project_per_layer_inputs(x, per_layer_inputs)
        # per_layer_inputs 形状: (batch_size, seq_len, n_layers, hidden_size_per_layer_input)

        if cache is not None:
            pos_start = self.current_pos
            pos_end = pos_start + input_ids.size(1)
            self.current_pos = pos_end
            mask_global, mask_local = self._create_masks(
                cur_len=input_ids.size(1),
                device=input_ids.device,
                pos_start=pos_start,
                pos_end=pos_end,
            )
            # 增量推理:mask 的 query 行只覆盖本次新增的 [pos_start, pos_end),但 key 列覆盖全部历史 [0, pos_end)
        else:
            pos_start = 0
            mask_global, mask_local = self._create_masks(
                cur_len=input_ids.size(1),
                device=input_ids.device,
                pos_start=0,
                pos_end=input_ids.size(1),
            )
            # 无 KV Cache 时,pos_start 恒为 0,query/key 都覆盖完整的 [0, seq_len)

        for i, block in enumerate(self.blocks):
            per_layer_input = per_layer_inputs[:, :, i, :] if per_layer_inputs is not None else None
            block_cache = cache.get(i) if cache is not None else None
            shared_kv = None
            if cache is not None and block.att.is_kv_shared_layer:
                shared_kv = cache.get(block.att.kv_shared_layer_index)
                # 命中跨层 KV 共享的层:不读取/更新自己的 block_cache,而是去取被共享层已经算好的 (key, value)
            x, next_cache = block(
                x,
                per_layer_input,
                mask_local,
                mask_global,
                self.cos_local,
                self.sin_local,
                self.cos_global,
                self.sin_global,
                start_pos=pos_start,
                cache=block_cache,
                shared_kv=shared_kv,
            )
            if cache is not None and next_cache is not None:
                cache.update(i, next_cache)
                # 把该层新的 (key, value) 写回 KV Cache,供下一次增量前向复用

        x = self.final_norm(x)
        logits = self.out_head(x)
        if self.cfg.get("final_logit_softcap") is not None:
            logits = logits / self.cfg["final_logit_softcap"]
            logits = torch.tanh(logits)
            logits = logits * self.cfg["final_logit_softcap"]
        return logits
        # final_logit_softcap: 用 tanh 对 logits 做“软上限截断”,把数值压缩到 (-softcap, softcap) 区间内,
        # 避免极端 logit 值导致训练/推理不稳定(Gemma2 引入、Gemma3 官方架构去掉、Gemma4 在此又重新使用)

    # reset_kv_cache: 开始新一轮生成前调用,把“已处理 token 计数”清零,
    # 避免复用上一次生成遗留下来的位置信息
    def reset_kv_cache(self):
        self.current_pos = 0


# 别名:与其它 standalone 笔记本的命名习惯保持一致,便于外部代码统一用 XxxModel 引用
Gemma4Model = Gemma4DenseModel

2. Initialize model

In [ ]:
# ==================== Gemma4 两档规模的配置字典:E2B(约 2B 有效参数)与 E4B(约 4B 有效参数) ====================
def get_gemma4_dense_config(model_size="E2B", dtype=torch.bfloat16):
    # model_size 大小写不敏感,统一转成大写再比较
    model_size = model_size.upper()

    if model_size == "E2B":
        # E2B: emb_dim=1536, 35 层, 8 个 Query 头但只有 1 个 KV 头(接近 Multi-Query Attention,
        # 极致压缩 KV Cache 显存);局部/全局按 4:1 的比例交替(每 5 层里 4 层滑窗、1 层全局)
        return {
            "vocab_size": 262_144,
            "vocab_size_per_layer_input": 262_144,
            "emb_dim": 1536,
            "hidden_dim": 4 * 1536,
            "n_layers": 35,
            "n_heads": 8,
            "head_dim": 256,
            "n_kv_heads": 1,
            "num_global_kv_heads": None,
            "global_head_dim": 512,
            "context_length": 131_072,
            "sliding_window": 512,
            "layer_types": (["sliding_attention"] * 4 + ["full_attention"]) * 7,
            # layer_types: 长度 35 的列表,元素为 "sliding_attention" 或 "full_attention",
            # 逐层显式指定注意力类型,(局部窗口*4 + 全局)重复 7 次 = 35 层
            "hidden_size_per_layer_input": 256,
            "num_kv_shared_layers": 20,
            # num_kv_shared_layers=20: 最后 20 层会跨层共享 KV(见 Gemma4Attention 中的判断逻辑),
            # 大幅降低长上下文推理时的显存与算力开销
            "use_double_wide_mlp": True,
            "attention_k_eq_v": False,
            "rope_local_base": 10_000.0,
            "rope_local_type": "default",
            "rope_global_base": 1_000_000.0,
            "rope_global_type": "proportional",
            "rope_global_partial_rotary_factor": 0.25,
            # rope_global_partial_rotary_factor=0.25: 全局层只对 25% 的维度做旋转位置编码,
            # 其余 75% 维度不加位置编码(NoPE),对应 compute_rope_params 里 rope_type="proportional" 分支
            "layer_norm_eps": 1e-6,
            "final_logit_softcap": 30.0,
            # final_logit_softcap=30.0: 最终 logits 会被 tanh 软上限截断到 (-30, 30) 区间
            "tie_word_embeddings": True,
            "pad_token_id": 0,
            "dtype": dtype,
        }

    # E4B: 更宽的 emb_dim=2560、更多层数 42、KV 头数增至 2,局部/全局按 5:1 交替,
    # 且不启用 double-wide MLP(use_double_wide_mlp=False),其余设计与 E2B 一致
    if model_size == "E4B":
        return {
            "vocab_size": 262_144,
            "vocab_size_per_layer_input": 262_144,
            "emb_dim": 2560,
            "hidden_dim": 4 * 2560,
            "n_layers": 42,
            "n_heads": 8,
            "head_dim": 256,
            "n_kv_heads": 2,
            "num_global_kv_heads": None,
            "global_head_dim": 512,
            "context_length": 131_072,
            "sliding_window": 512,
            "layer_types": (["sliding_attention"] * 5 + ["full_attention"]) * 7,
            # layer_types: (局部窗口*5 + 全局)重复 7 次 = 42 层
            "hidden_size_per_layer_input": 256,
            "num_kv_shared_layers": 18,
            "use_double_wide_mlp": False,
            "attention_k_eq_v": False,
            "rope_local_base": 10_000.0,
            "rope_local_type": "default",
            "rope_global_base": 1_000_000.0,
            "rope_global_type": "proportional",
            "rope_global_partial_rotary_factor": 0.25,
            "layer_norm_eps": 1e-6,
            "final_logit_softcap": 30.0,
            "tie_word_embeddings": True,
            "pad_token_id": 0,
            "dtype": dtype,
        }

    raise ValueError(f"Unknown Gemma 4 dense size: {model_size}")
# 按优先级选择可用的计算设备:CUDA > Apple Silicon(MPS) > CPU;
# 非 CUDA 设备统一退回 float32,因为 MPS/CPU 对 bfloat16 的算子支持/性能通常不如 CUDA
if torch.cuda.is_available():
    device = torch.device("cuda")
    model_dtype = torch.bfloat16
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    model_dtype = torch.float32
else:
    device = torch.device("cpu")
    model_dtype = torch.float32

# 先用 float32 构造一份配置字典,仅用于下面打印各项超参数(不代表最终加载模型时使用的精度,
# 真正加载权重时会在 cell 9 里用 model_dtype 重新调用 get_gemma4_dense_config)
selected_cfg = get_gemma4_dense_config(CHOOSE_MODEL, dtype=torch.float32)
# 裸表达式 {...}:在 Notebook 中会把这个字典渲染成输出,方便一眼确认当前选择的模型规模与关键超参数
{
    "model": CHOOSE_MODEL,
    "instruct": USE_INSTRUCT_MODEL,
    "emb_dim": selected_cfg["emb_dim"],
    "n_layers": selected_cfg["n_layers"],
    "n_heads": selected_cfg["n_heads"],
    "n_kv_heads": selected_cfg["n_kv_heads"],
    "global_head_dim": selected_cfg["global_head_dim"],
    "num_kv_shared_layers": selected_cfg["num_kv_shared_layers"],
    "device": str(device),
    "dtype": str(model_dtype),
}

3. Load pretrained weights

In [ ]:
# ==================== 将 HuggingFace 格式的 Gemma4 预训练权重加载进自定义模型 ====================
# 与很多 standalone 笔记本直接硬编码 HF 参数名前缀不同,这里做了更宽松的前缀探测
# ("model.language_model." / "language_model." / "model." / ""),以兼容不同发布形式的 checkpoint
def load_weights_into_gemma4_dense(model, cfg, params):
    # assign: 校验形状一致后,把右侧权重张量拷贝进左侧参数,保持左侧原有的 dtype/device 不变
    def assign(left, right, tensor_name="unknown"):
        if right is None:
            return False
        if left.shape != right.shape:
            raise ValueError(
                f"Shape mismatch in tensor {tensor_name!r}. Left: {tuple(left.shape)}, Right: {tuple(right.shape)}"
            )
        with torch.no_grad():
            if isinstance(right, torch.Tensor):
                left.copy_(right.to(dtype=left.dtype, device=left.device))
            else:
                left.copy_(torch.as_tensor(right, dtype=left.dtype, device=left.device))
        return True

    # 探测 params 里的 key 实际使用了哪种前缀(可能同时匹配多种,也可能都不匹配,
    # 后者会退回不加前缀直接查找)
    prefixes = []
    for prefix in ("model.language_model.", "language_model.", "model.", ""):
        if any(name.startswith(prefix) for name in params):
            prefixes.append(prefix)
    if not prefixes:
        prefixes = [""]

    # get_tensor: 依次尝试「不加前缀直接查找」和「各候选前缀 + 各候选名字」两种方式定位权重,
    # 返回命中的 (tensor, 实际使用的 key 名);全部未命中则返回 (None, None)
    def get_tensor(*names):
        for name in names:
            if name in params:
                return params[name], name
        for prefix in prefixes:
            for name in names:
                key = f"{prefix}{name}"
                if key in params:
                    return params[key], key
        return None, None

    loaded = 0
    missing = []

    # assign_from: 组合 get_tensor + assign,查不到就记录到 missing 列表,不立即报错
    # (等所有权重都尝试完,再统一判断是否存在遗漏,见下方 loaded==0 / missing 的处理)
    def assign_from(target, *names):
        nonlocal loaded
        tensor, name = get_tensor(*names)
        expected_name = names[0]
        if tensor is None:
            missing.append(expected_name)
            return
        loaded += int(assign(target, tensor, name or expected_name))

    # 词嵌入权重,形状为 (vocab_size, emb_dim)
    assign_from(model.tok_emb.weight, "embed_tokens.weight")

    # 逐层专属 embedding 相关权重(仅当模型启用 hidden_size_per_layer_input 时存在于 checkpoint 中)
    if getattr(model, "hidden_size_per_layer_input", 0):
        assign_from(model.embed_tokens_per_layer.weight, "embed_tokens_per_layer.weight")
        assign_from(model.per_layer_model_projection.weight, "per_layer_model_projection.weight")
        assign_from(model.per_layer_projection_norm.weight, "per_layer_projection_norm.weight")

    # 逐层拷贝:注意力投影、QKV-Norm、FeedForward(门控 MLP 的三个线性层)、
    # 四个 RMSNorm,以及可选的逐层输入相关权重与 layer_scalar
    for layer_idx in range(cfg["n_layers"]):
        block = model.blocks[layer_idx]
        prefix = f"layers.{layer_idx}."

        assign_from(block.att.q_proj.weight, f"{prefix}self_attn.q_proj.weight")
        assign_from(block.att.k_proj.weight, f"{prefix}self_attn.k_proj.weight")
        assign_from(block.att.v_proj.weight, f"{prefix}self_attn.v_proj.weight")
        assign_from(block.att.o_proj.weight, f"{prefix}self_attn.o_proj.weight")
        assign_from(block.att.q_norm.weight, f"{prefix}self_attn.q_norm.weight")
        assign_from(block.att.k_norm.weight, f"{prefix}self_attn.k_norm.weight")
        # 注意:v_norm 在自定义模型里是 with_scale=False(无权重),因此这里没有对应的
        # self_attn.v_norm.weight 赋值,与 q_norm/k_norm 不同

        # HF 命名与本实现的对应关系: gate_proj/up_proj/down_proj 与自身模块属性名一致,直接同名映射
        assign_from(block.mlp.gate_proj.weight, f"{prefix}mlp.gate_proj.weight")
        assign_from(block.mlp.up_proj.weight, f"{prefix}mlp.up_proj.weight")
        assign_from(block.mlp.down_proj.weight, f"{prefix}mlp.down_proj.weight")

        assign_from(block.input_layernorm.weight, f"{prefix}input_layernorm.weight")
        assign_from(block.post_attention_layernorm.weight, f"{prefix}post_attention_layernorm.weight")
        assign_from(block.pre_feedforward_layernorm.weight, f"{prefix}pre_feedforward_layernorm.weight")
        assign_from(block.post_feedforward_layernorm.weight, f"{prefix}post_feedforward_layernorm.weight")

        if getattr(block, "hidden_size_per_layer_input", 0):
            assign_from(block.per_layer_input_gate.weight, f"{prefix}per_layer_input_gate.weight")
            assign_from(block.per_layer_projection.weight, f"{prefix}per_layer_projection.weight")
            assign_from(block.post_per_layer_input_norm.weight, f"{prefix}post_per_layer_input_norm.weight")

        # layer_scalar 形状为 (1,),对应 Gemma4DenseBlock.__init__ 中初始化为全 1 的 buffer,
        # 这里若 checkpoint 中存在对应权重,会覆盖掉默认的全 1 初始值
        assign_from(block.layer_scalar, f"{prefix}layer_scalar")

    # 最终输出前的归一化层
    assign_from(model.final_norm.weight, "norm.weight")
    # 输出投影层;若权重是绑定的(tie_word_embeddings),lm_head.weight 可能不存在,
    # get_tensor 会自动退回用 embed_tokens.weight 兜底(assign_from 传入的第二个候选名字)
    assign_from(model.out_head.weight, "lm_head.weight", "embed_tokens.weight")

    # 完全没找到任何一个权重(说明前缀猜测彻底失败),直接报错,避免静默地用随机初始化权重推理
    if loaded == 0:
        raise KeyError(
            "No Gemma 4 language-model weights were loaded. Supported prefixes are "
            "'model.language_model.', 'language_model.', 'model.', and ''."
        )

    # 部分权重缺失但不是全部缺失:同样直接报错并列出前 10 个缺失的名字,便于排查 checkpoint 版本差异
    if missing:
        missing_preview = ", ".join(repr(name) for name in missing[:10])
        if len(missing) > 10:
            missing_preview += f", ... (+{len(missing) - 10} more)"
        raise KeyError(
            f"Missing {len(missing)} required Gemma 4 language-model tensors. "
            f"First missing tensors: {missing_preview}"
        )

    return loaded

load_weights_into_gemma4 = load_weights_into_gemma4_dense
# ==================== KVCache:按层存储的简单 K/V 缓存容器 ====================
# self.cache 是长度为 n_layers 的列表;每个元素要么是 None(尚未缓存),
# 要么是一个 (key, value) 元组,形状均为 (batch, num_kv_heads, cached_len, head_dim)
# (滑窗层的 cached_len 不会超过对应的 sliding_window,由 Gemma4DenseBlock.forward 负责截断)
class KVCache:
    def __init__(self, n_layers):
        self.cache = [None] * n_layers

    def get(self, layer_idx):
        return self.cache[layer_idx]

    def update(self, layer_idx, value):
        self.cache[layer_idx] = value

    def get_all(self):
        return self.cache

    # reset: 开始新一轮生成(新的 prompt / 新的 batch)前调用,清空所有层的缓存
    def reset(self):
        for i in range(len(self.cache)):
            self.cache[i] = None

In [ ]:
# 如果是第一次运行本 notebook,需要先登录 HuggingFace(Gemma 系列模型通常需要在 HF 上接受许可协议后才能下载)
# Uncomment and run the following code if you are executing the notebook for the first time

# from huggingface_hub import login
# login()
# ==================== 下载/定位权重文件,实例化模型并加载预训练权重 ====================
import json
from pathlib import Path
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download, snapshot_download

repo_id = f"google/gemma-4-{CHOOSE_MODEL}-it" if USE_INSTRUCT_MODEL else f"google/gemma-4-{CHOOSE_MODEL}"
# repo_id 形如 "google/gemma-4-E2B-it" 或 "google/gemma-4-E2B",取决于 CHOOSE_MODEL 与 USE_INSTRUCT_MODEL
local_dir_name = Path(repo_id).parts[-1]


# resolve_local_model_dir: 依次尝试几个常见的本地目录候选(当前目录 / 17_gemma4 子目录 / ch05/17_gemma4),
# 只要发现其中已经存在 model.safetensors 或 tokenizer.json,就直接复用,避免重复下载
def resolve_local_model_dir(local_dir_name):
    candidates = [
        Path(local_dir_name),
        Path("17_gemma4") / local_dir_name,
        Path("ch05") / "17_gemma4" / local_dir_name,
    ]
    for candidate in candidates:
        if (candidate / "model.safetensors").exists() or (candidate / "tokenizer.json").exists():
            return candidate
    return Path(local_dir_name)


local_dir = resolve_local_model_dir(local_dir_name)
# model_cfg 用真正的 model_dtype(前面按设备选定的 bfloat16 或 float32)重新构造配置字典,
# 与 cell 6 里仅用于打印展示、写死 float32 的 selected_cfg 不是同一份
model_cfg = get_gemma4_dense_config(CHOOSE_MODEL, dtype=model_dtype)
model = Gemma4DenseModel(model_cfg)
# 此时 model 的所有权重仍是随机初始化,下面才会用预训练权重覆盖

# 优先尝试单文件 safetensors(体积较小的模型,如 E2B/E4B 常打包成单个文件);
# 若不存在,再尝试用 hf_hub_download 单独下载该文件;仍失败(说明权重被切分成多个 shard)
# 则退回用 snapshot_download 拉取整个仓库,再按 index.json 里的 weight_map 逐个 shard 加载合并
single_file = Path(local_dir) / "model.safetensors"

if single_file.exists():
    weights_dict = load_file(single_file)
else:
    try:
        weights_path = hf_hub_download(
            repo_id=repo_id,
            filename="model.safetensors",
            local_dir=str(local_dir),
        )
        weights_dict = load_file(weights_path)
    except Exception:
        repo_dir = snapshot_download(repo_id=repo_id, local_dir=str(local_dir))
        index_path = Path(repo_dir) / "model.safetensors.index.json"
        with open(index_path, "r") as f:
            index = json.load(f)

        weights_dict = {}
        for filename in sorted(set(index["weight_map"].values())):
            shard = load_file(Path(repo_dir) / filename)
            weights_dict.update(shard)

num_loaded_tensors = load_weights_into_gemma4_dense(model, model_cfg, weights_dict)
# 用探测到的前缀与命名规则,把 weights_dict 中的张量逐一拷贝进 model 的对应参数
print(f"Using Gemma 4 files from: {local_dir}")
print(f"Loaded {num_loaded_tensors} Gemma 4 text tensors")

# 权重加载完成后再整体搬到目标计算设备(CUDA/MPS/CPU)
model.to(device)

# 释放原始权重字典,避免权重在内存里保留两份(model 参数 + weights_dict)
del weights_dict

# 切换到 eval 模式(关闭 dropout 等训练专属行为);裸表达式 model.eval() 同时会在输出区打印模型结构
model.eval()

4. Load tokenizer

In [ ]:
# ==================== GemmaTokenizer:基于 tokenizers 库封装的 Gemma 分词器 ====================
# 直接加载 HuggingFace 发布的 tokenizer.json(不依赖 transformers 的 AutoTokenizer),
# 保持本 notebook“standalone、不依赖 transformers”的定位
from tokenizers import Tokenizer


class GemmaTokenizer:
    def __init__(self, tokenizer_file_path):
        tok_file = Path(tokenizer_file_path)
        self._tok = Tokenizer.from_file(str(tok_file))

        # Gemma 对话格式中用到的一组特殊 token 字符串;注意 Gemma4 这里用的是单一的 <turn|> 标记
        # (区别于 Gemma3 用 <start_of_turn>/<end_of_turn> 两个不同的标记)
        self.bos_token = "<bos>"
        self.eos_token = "<eos>"
        self.pad_token = "<pad>"
        self.turn_token = "<turn|>"

        self.bos_token_id = self._tok.token_to_id(self.bos_token)
        self.eos_token_id = self._tok.token_to_id(self.eos_token)
        self.pad_token_id = self._tok.token_to_id(self.pad_token)
        self.turn_token_id = self._tok.token_to_id(self.turn_token)
        # 把上面的特殊 token 字符串转换为对应的 token id,方便后续拼接 / 判断

    # encode: 文本 -> token id 列表
    def encode(self, text, add_special_tokens=True):
        return self._tok.encode(text, add_special_tokens=add_special_tokens).ids

    # decode: token id(或 id 列表)-> 文本;skip_special_tokens 控制是否过滤掉 <bos>/<eos> 等特殊 token
    def decode(self, ids, skip_special_tokens=False):
        if isinstance(ids, int):
            ids = [ids]
        return self._tok.decode(ids, skip_special_tokens=skip_special_tokens)

    # apply_chat_template: 按 Gemma4 的对话格式拼接多轮消息,格式形如:
    #   <bos><turn|>user\n...内容...<turn|>\n<turn|>model\n...
    # 注意:OpenAI 风格里的 'assistant' 角色,这里要写成 'model'
    def apply_chat_template(self, messages, tokenize=False, add_generation_prompt=False):
        text = self.bos_token
        for message in messages:
            role = "model" if message["role"] == "assistant" else message["role"]
            text += f"{self.turn_token}{role}\n{message['content']}{self.turn_token}\n"

        if add_generation_prompt:
            text += f"{self.turn_token}model\n"

        if tokenize:
            return self.encode(text, add_special_tokens=False)
        return text
# 优先复用前面下载权重时已经落盘的 tokenizer.json;如果本地没有,再单独从 HF Hub 下载
tokenizer_file_path = Path(local_dir) / "tokenizer.json"
if not tokenizer_file_path.exists():
    try:
        tokenizer_file_path = Path(
            hf_hub_download(repo_id=repo_id, filename="tokenizer.json", local_dir=str(local_dir))
        )
    except Exception as e:
        print(f"Warning: failed to download tokenizer.json: {e}")

# 实例化分词器
tokenizer = GemmaTokenizer(tokenizer_file_path=str(tokenizer_file_path))
# 构造一个示例用户提问
prompt = "Give me a short introduction to large language models."

# 指令微调模型:套用 Gemma4 的对话模板(自动加 <bos>/<turn|> 等特殊标记),
# 并在末尾追加生成提示(model 角色起始标记),然后编码为 token id 列表
# (add_special_tokens=False 是因为 apply_chat_template 已经手动拼好了 <bos>,不需要 tokenizer 再加一次)
if USE_INSTRUCT_MODEL:
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
    input_token_ids = tokenizer.encode(prompt, add_special_tokens=False)
# 非指令微调模型:退回最朴素的续写格式
else:
    prompt = f"{prompt}\n\nAnswer:"
    input_token_ids = tokenizer.encode(prompt)

# 裸表达式:把编码后再解码回来的文本打印到输出区,用于肉眼检查模板拼接、特殊 token 是否符合预期
tokenizer.decode(input_token_ids, skip_special_tokens=False)

5. Generate text

In [ ]:
# Optionally use torch.compile for an extra speed-up
# model = torch.compile(model)
# ==================== 基于 KV Cache 的流式文本生成(贪心解码) ====================
# 与不带 KV Cache 的朴素生成相比,这里每一步只把“新产生的 1 个 token”喂给模型,
# 历史上下文的注意力信息由每一层内部的 KV Cache(以及跨层共享的 KV)承担,
# 避免了重复计算整段历史的开销
def generate_text_basic_stream(model, token_ids, max_new_tokens, eos_token_id=None, context_size=None):
    model.eval()

    with torch.no_grad():
        # 为每一层创建一个空的 KV Cache 容器,并把模型内部记录的位置计数器(current_pos)清零
        cache = KVCache(n_layers=model.cfg["n_layers"])
        model.reset_kv_cache()

        # Prime the cache with the initial context
        # 用完整的 prompt 做一次前向,把 prompt 对应的 K/V 写入每一层的缓存(“预热”/“填充” KV Cache),
        # 同时得到 prompt 最后一个位置的 logits,用于生成第一个新 token;
        # token_ids 形状: (batch_size, prompt_len);logits 形状: (batch_size, prompt_len, vocab_size)
        logits = model(token_ids, cache=cache)

        for _ in range(max_new_tokens):
            next_token = torch.argmax(logits[:, -1], dim=-1, keepdim=True)
            # 贪心解码:直接取概率最大的 token 作为下一个 token(未使用采样/温度/top-k/top-p 等策略);
            # next_token 形状: (batch_size, 1)

            if eos_token_id is not None and torch.all(next_token == eos_token_id):
                break
                # 命中终止 token(指令模型用 <turn|>,非指令模型用 <eos>)时停止生成

            yield next_token
            token_ids = torch.cat([token_ids, next_token], dim=1)
            # 这里的 token_ids 只用于函数外部记录已生成的完整序列,并不会整段重新喂给模型
            logits = model(next_token, cache=cache)
            # 关键点:只把新生成的 1 个 token(形状 (batch_size, 1))传给模型,
            # 模型内部会自动把它与各层 KV Cache 拼接后计算注意力,而不必重算历史部分
# 把编码好的 prompt token id 转成模型输入张量,形状为 (batch_size=1, prompt_len)
input_token_ids_tensor = torch.tensor(input_token_ids, device=device).unsqueeze(0)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

# 指令模型以 <turn|> 作为一轮对话结束的标志;非指令模型以标准的 <eos> 作为停止标志
stop_token_id = tokenizer.turn_token_id if USE_INSTRUCT_MODEL else tokenizer.eos_token_id

# 逐 token 调用生成器,一边生成一边把新 token 解码成文本并打印出来(流式输出效果)
for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=200,
    eos_token_id=stop_token_id,
):
    token_id = token.squeeze(0).tolist()
    print(tokenizer.decode(token_id), end="", flush=True)

# 生成结束后,如果在 CUDA 设备上运行,打印本次生成过程中峰值显存占用,便于评估 KV Cache(含跨层共享)的显存开销
if torch.cuda.is_available():
    def calc_gpu_gb(x):
        return f"{x / 1024 / 1024 / 1024:.2f} GB"

    print(f"\n\nGPU memory used: {calc_gpu_gb(torch.cuda.max_memory_allocated())}")